# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [58]:
#importing the packages that are required for this homework
import Bio.SeqIO
import Bio.Seq
import re

# defining test cases that include positives for HA and NA including the barcode, and then a negative case without the correct sequence
test_cases = [
    (Bio.Seq.Seq("CCGGATTTGCATATAATGATGCACCATAGGCGGCCGC" + "A"*16).reverse_complement(),("HA", "A"*16), "HA positive"),
    (Bio.Seq.Seq("CACGATAGATAAATAATAGTGCACCATAGGCGGCCGC" + "C"*16).reverse_complement(), ("NA", "C"*16),"NA positive"),
    ("CAGTCAGTCAGTCAGTCGAT", None, "Negative: no matching sequence")
]

def run_tests(parser_function, test_cases): #this does not have to change. loops test cases and passes it into parser function
    """Run a parser against a list of test cases and report the results.
    Args:
        parser_function: The function to test. It takes a sequence and returns a barcode
            string, or None.
        test_cases: A list of (sequence, expected result, label) tuples.
    Returns:
        The number of test cases that failed.
    """
    n_failed = 0 
    for seq, expected, label in test_cases:
        result = parser_function(seq)
        if result == expected:
            print(f"PASS  {label}")
        else:
            n_failed += 1
            print(f"FAIL  {label}: {seq!r} gave {result!r}, expected {expected!r}")

    return f"{n_failed} tests failed"

def read_barcode_gene(seqread): #defining function
    # defining HA and NA and upstream (I moved upsptream here for ease of formatting my re.compile statement)
    HA = "CCGGATTTGCATATAATGATGCACCAT"
    NA = "CACGATAGATAAATAATAGTGCACCAT"
    upstream = "AGGCGGCCGC"

    #Converting sequence into correct format (uppercase, then reverse complement)
    seqread_upper = str(seqread.upper())
                         
    seqread_original = str(Bio.Seq.Seq(seqread_upper).reverse_complement())

    #look for HA pattern
    HA_pattern = re.compile(HA + upstream + r"(?P<barcode>[ACGTN]{16})")

    #look for NA pattern
    NA_pattern = re.compile(NA + upstream + r"(?P<barcode>[ACGTN]{16})")

    #creating varaibles match the the pattern
    HA_match = HA_pattern.search(seqread_original)
    NA_match = NA_pattern.search(seqread_original)

    #conditional statement that says that a match must contain the HA/NA gene and the barcode
    if HA_match:
        return ("HA", HA_match.group("barcode"))

    elif NA_match:
        return ("NA", NA_match.group("barcode"))

    else:
        return None

#running my test cases
run_tests(read_barcode_gene, test_cases)

#defining seqreads as the fastq file that contains the 10,000 reads
seqreads = Bio.SeqIO.parse("barcodes_R1.fastq", "fastq")

#start counts for HA and NA containing reads
HA_count = 0
NA_count = 0

#creates a for loop that takes the seq data, analyses the content, and adds to the HA count if it contains HA and vice versa for NA
for record in seqreads:
    result = read_barcode_gene(record.seq)
    if result is not None:
        gene, barcode = result
        if gene == "HA":
            HA_count += 1
        elif gene == "NA":
            NA_count += 1

#prints the results
print("Reads with HA: ", HA_count)
print("Reads with NA:", NA_count)


PASS  HA positive
PASS  NA positive
PASS  Negative: no matching sequence
Reads with HA:  5250
Reads with NA: 3912


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [ ]:
test_cases = [
    ("CCGGATTTGCATATAATGATGCACCAT","HA", "HA positive"), ("CACGATAGATAAATAATAGTGCACCAT", "NA", "NA positive"), 
    ("CAGTCAGTCAGTCAGTCGAT", None, "Negative: no matching sequence")
]

def read_HAorNA(seqread): #this function looks for all reads that contain HA or NA regardless of having the barcode 
    
    HA = "CCGGATTTGCATATAATGATGCACCAT"
    NA = "CACGATAGATAAATAATAGTGCACCAT"
     
    seqread = str(seqread).upper()
    seqread = str(Bio.Seq.Seq(seqread).reverse_complement())
     
    HA_match = re.search(HA, seqread)
    NA_match = re.search(NA, seqread)

    if HA_match:
        return "HA"
    elif NA_match:
        return "NA"
    else:
        return None

#run_tests(read_HAorNA, test_cases)

seqreads = Bio.SeqIO.parse("barcodes_R1.fastq", "fastq")

#start counts for HA and NA containing reads
HA_count = 0
NA_count = 0

for record in seqreads:
    result = read_HAorNA(record.seq)
    if result == "HA":
        HA_count += 1
    elif result == "NA":
        NA_count += 1

print("HA:", HA_count)
print("NA:", NA_count)

HA: 5409
NA: 4122


In [ ]:
#this gives the amount without barcodes. I subtract the total amount of HA and NA instances from those with barcodes
print("HA without barcode:", 5409-5250)
print("NA without barcode:", 4121-3912)

HA without barcode: 159
NA without barcode: 209


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [66]:
# create dictionaries
HA_barcodes = {}
NA_barcodes = {}

seqreads = Bio.SeqIO.parse("barcodes_R1.fastq", "fastq")

for record in seqreads:
    result = read_barcodeHANA(record.seq)
    if result is not None:
        gene, barcode = result
        if barcode is not None:
            if gene == "HA":
                HA_barcodes[barcode] = HA_barcodes.get(barcode, 0) + 1
            elif gene == "NA":
                NA_barcodes[barcode] = NA_barcodes.get(barcode, 0) + 1

HA_top_barcode = max(HA_barcodes, key=HA_barcodes.get)
NA_top_barcode = max(NA_barcodes, key=NA_barcodes.get)

print("Most common HA barcode:", HA_top_barcode)
print("HA barcode counts:", HA_barcodes[HA_top_barcode])

print("Most common NA barcode:", NA_top_barcode)
print("NA barcode counts:", NA_barcodes[NA_top_barcode])

Most common HA barcode: CCCGACCCGACATTAA
HA barcode counts: 155
Most common NA barcode: ACCAGTTCTCCCCGGG
NA barcode counts: 152
